# FLAC Benchmark Results

Loads and plots performance data from the Rust (Criterion) and C (libFLAC) benchmarks.

**Run benchmarks first:**
```bash
# Rust
cargo bench --features flac --bench flac_rw

# C
cd benches/C && make bench_flac && ./bench_flac
```

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd

REPO_ROOT = Path(".")  # notebook lives at repo root
CRITERION_DIR = REPO_ROOT / "target" / "criterion"
C_CSV = REPO_ROOT / "benches" / "C" / "flac_bench.csv"

plt.rcParams.update(
    {
        "figure.dpi": 130,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "grid.alpha": 0.3,
    }
)

In [ ]:
def load_criterion_flac() -> pd.DataFrame:
    """Walk target/criterion and collect all flac_read / flac_write results."""
    rows = []
    for bench_json in CRITERION_DIR.rglob("new/benchmark.json"):
        try:
            bm = json.loads(bench_json.read_text())
        except Exception:
            continue

        group = bm.get("group_id", "")
        if not group.startswith("flac_"):
            continue

        estimates_path = bench_json.parent / "estimates.json"
        if not estimates_path.exists():
            continue
        est = json.loads(estimates_path.read_text())

        mean_ns = est["mean"]["point_estimate"]
        stddev_ns = est["std_dev"]["point_estimate"]
        throughput_bytes = (bm.get("throughput") or {}).get("Bytes", None)

        mbps = None
        if throughput_bytes:
            mbps = (throughput_bytes / (1024**2)) / (mean_ns / 1e9)

        rows.append(
            {
                "source": "audio_samples_io"
                if bm["function_id"].startswith("audio")
                else bm["function_id"].split("-")[0],
                "group": group,
                "bench_id": bm["function_id"],
                "case": bm.get("value_str", ""),
                "mean_ns": mean_ns,
                "stddev_ns": stddev_ns,
                "throughput_bytes": throughput_bytes,
                "MBps": mbps,
            }
        )

    df = pd.DataFrame(rows)
    if df.empty:
        print("⚠  No Criterion FLAC results found — run the Rust bench first.")
        return df

    # Parse sample_rate and channels out of case label e.g. "44100hz_2ch"
    df["sample_rate"] = df["case"].str.extract(r"(\d+)hz").astype(float)
    df["channels"] = df["case"].str.extract(r"(\d+)ch").astype(float)
    df["mean_us"] = df["mean_ns"] / 1_000
    return df


def load_c_flac() -> pd.DataFrame:
    """Load the C libFLAC benchmark CSV."""
    if not C_CSV.exists():
        print(f"⚠  {C_CSV} not found — run the C bench first.")
        return pd.DataFrame()

    df = pd.read_csv(C_CSV)
    df.columns = df.columns.str.strip()
    df["source"] = "libFLAC (C)"
    df["mean_us"] = df["mean_ns"] / 1_000
    df["sample_rate"] = df["case_label"].str.extract(r"(\d+)hz").astype(float)
    df["channels"] = df["case_label"].str.extract(r"(\d+)ch").astype(float)
    df = df.rename(columns={"case_label": "case"})
    return df


rust = load_criterion_flac()
c = load_c_flac()
print(f"Rust rows: {len(rust)}  |  C rows: {len(c)}")

In [ ]:
# Unified dataframe: align column names and concatenate
def unify(rust_df: pd.DataFrame, c_df: pd.DataFrame) -> pd.DataFrame:
    keep = [
        "source",
        "group",
        "bench_id",
        "case",
        "mean_ns",
        "mean_us",
        "stddev_ns",
        "MBps",
        "sample_rate",
        "channels",
    ]

    parts = []
    if not rust_df.empty:
        parts.append(rust_df[[c for c in keep if c in rust_df.columns]])
    if not c_df.empty:
        c_df2 = c_df.copy()
        if "bench_id" not in c_df2.columns:
            c_df2["bench_id"] = (
                c_df2["bench_id"] if "bench_id" in c_df2.columns else c_df2["bench_id"]
            )
        parts.append(c_df2[[c for c in keep if c in c_df2.columns]])

    if not parts:
        return pd.DataFrame(columns=keep)
    return pd.concat(parts, ignore_index=True)


all_df = unify(rust, c)
all_df.head(10)

## Read throughput (MiB/s) — all implementations

In [ ]:
def plot_throughput_by_case(df: pd.DataFrame, group: str, title: str):
    data = df[(df["group"] == group) & df["MBps"].notna()].copy()
    if data.empty:
        print(f"No data for group '{group}'")
        return

    cases = sorted(data["case"].unique())
    impls = sorted(data["bench_id"].unique())
    x = np.arange(len(cases))
    width = 0.8 / max(len(impls), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(cases) * 1.4), 5))

    for i, impl in enumerate(impls):
        sub = data[data["bench_id"] == impl].set_index("case")
        vals = [sub.loc[c, "MBps"] if c in sub.index else 0.0 for c in cases]
        offset = (i - len(impls) / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width * 0.9, label=impl)
        for bar, v in zip(bars, vals):
            if v > 0:
                ax.text(
                    bar.get_x() + bar.get_width() / 2,
                    bar.get_height() + 0.5,
                    f"{v:.0f}",
                    ha="center",
                    va="bottom",
                    fontsize=7,
                    rotation=45,
                )

    ax.set_xticks(x)
    ax.set_xticklabels(cases, rotation=30, ha="right")
    ax.set_ylabel("Throughput (MiB/s)")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


plot_throughput_by_case(all_df, "flac_read", "FLAC Read — throughput (MiB/s) by case")
plot_throughput_by_case(all_df, "flac_write", "FLAC Write — throughput (MiB/s) by case")

## Latency (µs) — all implementations

In [ ]:
def plot_latency_by_case(df: pd.DataFrame, group: str, title: str):
    data = df[(df["group"] == group) & df["mean_us"].notna()].copy()
    if data.empty:
        print(f"No data for group '{group}'")
        return

    cases = sorted(data["case"].unique())
    impls = sorted(data["bench_id"].unique())
    x = np.arange(len(cases))
    width = 0.8 / max(len(impls), 1)

    fig, ax = plt.subplots(figsize=(max(10, len(cases) * 1.4), 5))

    for i, impl in enumerate(impls):
        sub = data[data["bench_id"] == impl].set_index("case")
        vals = [sub.loc[c, "mean_us"] if c in sub.index else 0.0 for c in cases]
        errs = []
        for c in cases:
            if c in sub.index and "stddev_ns" in sub.columns:
                errs.append(sub.loc[c, "stddev_ns"] / 1_000)
            else:
                errs.append(0.0)
        offset = (i - len(impls) / 2 + 0.5) * width
        ax.bar(x + offset, vals, width * 0.9, yerr=errs, capsize=3, label=impl)

    ax.set_xticks(x)
    ax.set_xticklabels(cases, rotation=30, ha="right")
    ax.set_ylabel("Mean latency (µs)")
    ax.set_title(title)
    ax.legend(loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


plot_latency_by_case(all_df, "flac_read", "FLAC Read — mean latency (µs)")
plot_latency_by_case(all_df, "flac_write", "FLAC Write — mean latency (µs)")

## Throughput vs channel count

In [ ]:
def plot_scaling_by_channels(
    df: pd.DataFrame, group: str, sample_rate: float, title: str
):
    data = df[
        (df["group"] == group) & (df["sample_rate"] == sample_rate) & df["MBps"].notna()
    ].copy()
    if data.empty:
        print(f"No data for {group} @ {sample_rate:.0f} Hz")
        return

    fig, ax = plt.subplots(figsize=(8, 4))
    for impl, grp in data.groupby("bench_id"):
        grp_sorted = grp.sort_values("channels")
        ax.plot(grp_sorted["channels"], grp_sorted["MBps"], marker="o", label=impl)

    ax.set_xlabel("Channels")
    ax.set_ylabel("Throughput (MiB/s)")
    ax.set_title(title)
    ax.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()


for sr in [44100.0, 96000.0]:
    plot_scaling_by_channels(
        all_df, "flac_read", sr, f"FLAC Read scaling — {sr:.0f} Hz"
    )
    plot_scaling_by_channels(
        all_df, "flac_write", sr, f"FLAC Write scaling — {sr:.0f} Hz"
    )

## Rust vs C: relative speed (audio_samples_io / libFLAC)

In [ ]:
def plot_relative_speed(
    rust_df: pd.DataFrame, c_df: pd.DataFrame, group: str, title: str
):
    if rust_df.empty or c_df.empty:
        print("Need both Rust and C results for relative plot.")
        return

    # Use our i16 impl as the Rust reference, libflac-i16 as C reference
    rust_ref = rust_df[
        (rust_df["group"] == group) & (rust_df["bench_id"] == "audio-i16")
    ].set_index("case")["mean_ns"]
    c_ref = c_df[
        (c_df["group"] == group) & (c_df["bench_id"] == "libflac-i16")
    ].set_index("case")["mean_ns"]

    common = rust_ref.index.intersection(c_ref.index)
    if common.empty:
        print("No common cases between Rust and C results.")
        return

    ratio = (rust_ref[common] / c_ref[common]).sort_index()

    fig, ax = plt.subplots(figsize=(max(8, len(common) * 1.2), 4))
    colors = ["#d62728" if r > 1 else "#2ca02c" for r in ratio]
    bars = ax.bar(range(len(ratio)), ratio.values, color=colors)
    ax.axhline(
        1.0, color="black", linewidth=1, linestyle="--", label="parity with libFLAC"
    )
    ax.set_xticks(range(len(ratio)))
    ax.set_xticklabels(ratio.index, rotation=30, ha="right")
    ax.set_ylabel("Latency ratio (audio_samples_io / libFLAC)")
    ax.set_title(title)
    for bar, v in zip(bars, ratio.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{v:.2f}×",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print(
        f"Geometric mean ratio: {np.exp(np.log(ratio).mean()):.2f}×  "
        f"(>1 = slower than libFLAC, <1 = faster)"
    )


plot_relative_speed(
    rust, c, "flac_read", "Read: audio_samples_io vs libFLAC (i16, lower = faster)"
)
plot_relative_speed(
    rust, c, "flac_write", "Write: audio_samples_io vs libFLAC (i16, lower = faster)"
)

## Rust vs Symphonia: relative speed

In [ ]:
def plot_rust_vs_symphonia(df: pd.DataFrame, group: str, title: str):
    audio = df[(df["group"] == group) & (df["bench_id"] == "audio-i16")].set_index(
        "case"
    )["mean_ns"]
    symphonia = df[
        (df["group"] == group) & (df["bench_id"] == "symphonia-i32")
    ].set_index("case")["mean_ns"]

    common = audio.index.intersection(symphonia.index)
    if common.empty:
        print("No symphonia results found — run the Rust bench first.")
        return

    ratio = (audio[common] / symphonia[common]).sort_index()

    fig, ax = plt.subplots(figsize=(max(8, len(common) * 1.2), 4))
    colors = ["#d62728" if r > 1 else "#2ca02c" for r in ratio]
    bars = ax.bar(range(len(ratio)), ratio.values, color=colors)
    ax.axhline(
        1.0, color="black", linewidth=1, linestyle="--", label="parity with symphonia"
    )
    ax.set_xticks(range(len(ratio)))
    ax.set_xticklabels(ratio.index, rotation=30, ha="right")
    ax.set_ylabel("Latency ratio (audio_samples_io / symphonia)")
    ax.set_title(title)
    for bar, v in zip(bars, ratio.values):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.01,
            f"{v:.2f}×",
            ha="center",
            va="bottom",
            fontsize=8,
        )
    ax.legend(fontsize=8)
    plt.tight_layout()
    plt.show()

    print(
        f"Geometric mean ratio: {np.exp(np.log(ratio).mean()):.2f}×  "
        f"(>1 = slower than symphonia, <1 = faster)"
    )


plot_rust_vs_symphonia(
    rust, "flac_read", "Read: audio_samples_io vs Symphonia (lower = faster)"
)

## Summary table

In [ ]:
summary = (
    all_df.groupby(["group", "bench_id"])
    .agg(
        mean_latency_us=("mean_us", "mean"),
        mean_MBps=("MBps", "mean"),
        cases=("case", "count"),
    )
    .round(2)
)

summary = summary.sort_values(["group", "mean_latency_us"])
summary